# Corrected disagreement-aware analysis

Recomputes the released disagreement-aware summaries from the public OOF
probabilities.

In [ ]:
from pathlib import Path

def find_repo_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "gold").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Run the notebook from inside the repository."
    )

ROOT = find_repo_root()

import numpy as np
import pandas as pd
pred = pd.read_csv(ROOT / 'results' / 'predictions' / 'disagreement_aware_oof_predictions.csv')
pred.head()

In [ ]:
def soft_metrics(q, p):
    p = np.clip(p, 1e-12, 1)
    p = p / p.sum(axis=1, keepdims=True)
    q = q / q.sum(axis=1, keepdims=True)
    return {
        'cross_entropy': float(np.mean(-np.sum(q * np.log(p), axis=1))),
        'brier': float(np.mean(np.sum((p - q) ** 2, axis=1))),
        'human_label_probability': float(np.mean(np.sum(q * p, axis=1))),
    }
rows = []
for (task, regime, seed), group in pred.groupby(['task', 'regime', 'seed']):
    labels = [-1, 0, 1] if task == 'acceptance_3' else [0, 1, 2]
    row = {'task': task, 'regime': regime, 'seed': seed}
    row.update(soft_metrics(group[[f'q_{x}' for x in labels]].to_numpy(), group[[f'p_{x}' for x in labels]].to_numpy()))
    rows.append(row)
summary = pd.DataFrame(rows)
display(summary.groupby(['task', 'regime']).agg(['mean', 'std']))

In [ ]:
display(pd.read_csv(ROOT / 'results' / 'tables' / 'disagreement_aware_corrected.csv'))